In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms.v2 as transforms_v2

import csv
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import os
import tempfile
from pathlib import Path
import atexit
import glob
import shutil

import time
from tqdm import tqdm

from src.model_utils import setup_model
from src.config_utils import setup_config_variables
from src.data_utils import prepare_dataloader

In [ ]:
import torch.distributed as dist
import os

# Set up environment variables for a single-GPU "distributed" setup
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'  
os.environ['WORLD_SIZE'] = '1'
os.environ['RANK'] = '0'
os.environ['LOCAL_RANK'] = '0'

# Initialize the process group
dist.init_process_group(backend='nccl', init_method='env://')

In [ ]:
config = setup_config_variables()
BCTHW_MODELS = ["Simple", "Swin", "MViTV2", "MViTV2Parallel","R3d", "R3dParallel", "R2Plus1d", "R2Plus1dCBAM"]
NO_TIME_DIM_MODELS = ["Resnet", "ResnetCBAM", "ConvNeXt", "DRFuser"]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

print("This can be run after preprocessing.")

In [ ]:
# Delete old images
try:
        os.mkdir(config["IMAGES_DIR"])
except FileExistsError:
    for img in glob.glob(config["IMAGES_DIR"] + "/*.svg"):
        os.remove(img)

In [ ]:
# SETUP MODEL
model = setup_model(config["MODEL_TYPE"], config["MODEL_DIMS"], device)

In [ ]:
model_files = []
best_model_dir = f"{config['MODEL_SAVE_DIR']}/best/*.pth"
last_model_dir = f"{config['MODEL_SAVE_DIR']}/last/*.pth"

# pull best and last from a copy from tmpdir, in case they change while plotting (i.e. training is going on in parallel)
tmpdir = tempfile.mkdtemp()
atexit.register(lambda: shutil.rmtree(tmpdir, ignore_errors=True))
for path in glob.glob(best_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))

# Only take best model
#for path in glob.glob(last_model_dir):
#    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
    
# first best, then last
model_files.extend(sorted(glob.glob(tmpdir + "/*.pth"), key=os.path.getctime))

print(f"{model_files}")
train_files = sorted(glob.glob(config['TRAINING_DIR'] + "/*.hdf5"))
test_files = sorted(glob.glob(config['TEST_DIR'] + "/*.hdf5"))
whole_files = sorted(glob.glob(config['OUT_DIR'] + "/*.hdf5"))

In [ ]:
def reshuffle_data_for_model(data, model_type):
    b, t, c, h, w = data.shape
    # Prepare input based on model type
    if model_type in NO_TIME_DIM_MODELS:
        data = data.reshape((b * t, c, h, w))

    elif config["MODEL_TYPE"] in BCTHW_MODELS:
        data = data.permute(0, 2, 1, 3, 4)
    
    return data

In [ ]:
def visualize_losses(log_dir):
    log_file = f"{log_dir}/loss_log.csv"
    epochs = []
    train_losses = []
    test_losses = []

    with open(log_file, "r") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            epoch, train_loss, test_loss = int(row[0]), float(row[1]), float(row[2])
            epochs.append(epoch)
            train_losses.append((train_loss ** 0.5) * 180)
            test_losses.append((test_loss ** 0.5) * 180)

    print(f"min. test loss: {min(test_losses)}\nmin. train.loss: {min(train_losses)}")
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label="Train Loss", marker='o', color="blue")
    plt.plot(epochs, test_losses, label="Test Loss", marker='x', color="red")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Testing Loss")
    plt.legend()
    plt.grid(True)
    plt.ylim(bottom=0, top=max(70, max(max(train_losses), max(test_losses))))
    plt.show()

visualize_losses(config["MODEL_SAVE_DIR"])

In [ ]:
#setup default dataloaders
train_loader, test_loader = prepare_dataloader(config["TRAINING_DIR"], config["TEST_DIR"], config["BATCH_SIZE"], config["AUGMENT_DATA"])

# From Claude 3.7 Sonnet Thought
def measure_inference_time(model, test_file, batch_size=config["BATCH_SIZE"],  window_size=config["WINDOW_SIZE"], num_runs=10):
    model.eval()
    print("Warming up GPU...")
    with torch.no_grad():
        for i, (combined_images, steering_angle) in enumerate(test_loader):
            combined_images = combined_images.to(device)
            if i >= 10:
                break
            combined_images = reshuffle_data_for_model(combined_images, config["MODEL_TYPE"])
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                _ = model(combined_images)

    print(f"Measuring inference time over {num_runs} runs...")
    inference_times = []
    with torch.no_grad():
            pbar = tqdm(test_loader, total=100)
            for i, (combined_images, steering_angles) in enumerate(pbar):
                if i >= 100:
                    break
                combined_images = combined_images.to(device)
                combined_images = reshuffle_data_for_model(combined_images, config["MODEL_TYPE"])
                torch.cuda.synchronize() if torch.cuda.is_available() else None
                start_time = time.time()
                
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    _ = model(combined_images)
                    
                torch.cuda.synchronize() if torch.cuda.is_available() else None
                end_time = time.time()
                inference_times.append(end_time - start_time)
    
            
    # Calculate statistics
    total_frames = batch_size * window_size
    inference_times_array = np.array(inference_times)
    avg_time = np.mean(inference_times_array)
    std_time = np.std(inference_times_array)
    
    stats = {
        "avg_time_per_batch_ms": avg_time * 1000,
        "std_time_ms": std_time * 1000,
        "avg_time_per_frame_ms": (avg_time * 1000) / total_frames,
        "frames_per_second": total_frames / avg_time,
        "batch_size": batch_size,
        "window_size": window_size,
        "total_frames": total_frames
    }
    return stats, inference_times_array

In [ ]:
model_dict = torch.load(model_files[0], weights_only=False, map_location=device)
new_state_dict = {key.replace("module.", ""): value for key, value in model_dict["model_state_dict"].items()}
model.load_state_dict(new_state_dict)
model.to(device)

# Choose a representative test file
test_file = test_files[1]

stats, inference_times = measure_inference_time(model, test_file,num_runs=100)
print("\nInference Time Statistics:")
print(f"Model: {config['MODEL_TYPE']}")
print(f"Device: {device}")
print(f"Batch size: {stats['batch_size']}, Window size: {stats['window_size']}")
print(f"Average time per batch: {stats['avg_time_per_batch_ms']:.2f} ms (±{stats['std_time_ms']:.2f} ms)")
print(f"Average time per frame: {stats['avg_time_per_frame_ms']:.2f} ms")
print(f"Throughput: {stats['frames_per_second']:.2f} frames per second")

# Plot timing distribution
plt.figure(figsize=(10, 6))
plt.hist(inference_times * 1000, bins=20, alpha=0.7, color='blue')
plt.axvline(stats['avg_time_per_batch_ms'], color='red', linestyle='dashed', linewidth=2)
plt.xlabel('Inference Time (ms)')
plt.ylabel('Frequency')
plt.title('Distribution of Inference Times')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(config["IMAGES_DIR"], "inference_time_distribution.svg"), format="svg")
plt.show()

In [ ]:
criterion = nn.MSELoss()
def calculate_metrics(gt, pred):
    """Calculate RMSE and EVA for given ground truth and predictions."""
    gt_tensor = torch.tensor(gt, dtype=torch.float32).to(device)
    pred_tensor = torch.tensor(pred, dtype=torch.float32).to(device)
    
    loss = criterion(pred_tensor, gt_tensor)
    rmse = loss ** 0.5
    
    var_y = np.var(gt)
    eva = 1 - np.var(gt - pred) / var_y if var_y != 0 else 0
    
    return rmse.item(), eva

In [ ]:
def generate_predictions(model, file_path, batch_size=config["BATCH_SIZE"], window_size=config["WINDOW_SIZE"]):
    model.eval()
    with h5py.File(file_path, "r") as f:
        combined_transformed_frames, steering_angles = get_combined_transformed_frames_angles(file_path)
        combined_transformed_frames = truncate_data_to_window(combined_transformed_frames, window_size=window_size)
        steering_angles = truncate_data_to_window(steering_angles, window_size=window_size)
        
        preds, gts = [], []
        b_t = batch_size * window_size
        data_len = len(combined_transformed_frames)
        for start in range(0, data_len, b_t):
            end = min(start + b_t, data_len)
            batch_data = combined_transformed_frames[start:end] # b * t, c, h, w            
            batch_data = reshuffle_data_for_model(batch_data, config["MODEL_TYPE"], window_size=window_size, batch_size=batch_size)
            batch_data = batch_data.to(device)

            with torch.no_grad():
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    batch_out = model(batch_data).cpu().float().numpy()
                    batch_out = batch_out.flatten()
                
                preds.append(batch_out)
                if config["MODEL_TYPE"] not in NO_TIME_DIM_MODELS:
                    gts.append(steering_angles[start + window_size - 1:end:window_size])
                else:
                    gts.append(steering_angles[start:end])
       
        preds = np.concatenate(preds, axis=0)
        gts = np.concatenate(gts, axis=0)

        return preds, gts

In [ ]:
def process_file_predictions(model, file_path, running_stats_orig, running_stats_stretched, window_size=config["WINDOW_SIZE"]):
    """Process a single file for predictions and update running statistics."""
    _, steering_angles = get_combined_transformed_frames_angles(file_path)
    steering_angles = truncate_data_to_window(steering_angles, window_size=window_size).cpu().numpy()

    preds, gts = generate_predictions(model, file_path)
    rmse_orig, eva_orig = calculate_metrics(gts, preds)
    running_stats_orig.update(gts, preds)
    
    if config["MODEL_TYPE"] in NO_TIME_DIM_MODELS:
        print("This is a model with no time dimensions, no stretched models are computed")
        rmse_stretched, eva_stretched = float("inf"),  float("inf")
        preds_stretched = preds
        
    # Create stretched predictions
    else:
        preds_stretched = np.repeat(preds, window_size)
        rmse_stretched, eva_stretched = calculate_metrics(steering_angles, preds_stretched)
        running_stats_stretched.update(steering_angles, preds_stretched)

    return steering_angles, preds_stretched, {
        'rmse_orig': rmse_orig,
        'eva_orig': eva_orig,
        'rmse_stretched': rmse_stretched,
        'eva_stretched': eva_stretched
    }

In [ ]:
class RunningStats:
    """Track running statistics for explained variance and RMSE calculation."""
    def __init__(self):
        self.n_samples = 0
        self.sum_gt = 0.0
        self.sum_sq_gt = 0.0
        self.sum_diff_sq = 0.0
    
    def update(self, gt, pred):
        """Update running statistics with new batch of data."""
        if torch.is_tensor(gt):
            gt = gt.cpu().numpy()
        if torch.is_tensor(pred):
            pred = pred.cpu().numpy()
            
        diff = gt - pred
        self.n_samples += len(gt)
        self.sum_gt += np.sum(gt)
        self.sum_sq_gt += np.sum(np.square(gt))
        self.sum_diff_sq += np.sum(np.square(diff))
    
    def get_eva(self):
        """Calculate explained variance from running statistics."""
        if self.n_samples == 0:
            return 0
        
        mean_gt = self.sum_gt / self.n_samples
        var_gt = (self.sum_sq_gt / self.n_samples) - (mean_gt ** 2)
        var_diff = self.sum_diff_sq / self.n_samples
        
        return 1 - (var_diff / var_gt) if var_gt > 0 else 0
        
    def get_rmse(self):
        """Calculate RMSE from running statistics."""
        if self.n_samples == 0:
            return 0
        
        mse = self.sum_diff_sq / self.n_samples
        return np.sqrt(mse)

In [ ]:
def plot_predictions(model, files, model_files, dataset_name):
    fig = plt.figure(figsize=(15, 7.5 * len(files)))
    
    # Initialize running statistics
    running_stats_orig = RunningStats()
    running_stats_stretched = RunningStats()
    
    for i, file_path in enumerate(files):
        announce_str = f"Working on {file_path}, File [{i + 1}/{len(files)}]"
        print(f"{announce_str}\n{'-' * len(announce_str)}")
        
        ax = plt.subplot(len(files), 1, i + 1)
        ax.set_title(file_path, fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel("Predicted Values", fontsize=12)
        ax.grid(True)
        ax.set_ylim(-90, 90)
        ax.xaxis.set_major_locator(MultipleLocator(25))
        ax.tick_params(axis='x', rotation=45)
        
        with h5py.File(file_path, "r") as f5:
            gt = np.array(f5["steering_wheel_angle"])
            
        time_axis = [x * (config["BINSIZE_MS"] / 1000) for x in range(len(gt))]
        ax.plot(time_axis, gt, label="truth", linewidth=2, linestyle='--')
        
        for model_file in model_files:
            epoch = Path(model_file).stem.split('_')[0]
            epoch = epoch.split('/')[-1][1:]
            
            # Load model
            loc = f"cuda:{0}"
            model_dict = torch.load(model_file, weights_only=False, map_location=loc)
            new_state_dict = {k.replace("module.", ""): v for k, v in model_dict["model_state_dict"].items()}
            model.load_state_dict(new_state_dict)
            model.to(device)
            
            # Process file predictions
            gt, preds_stretched, metrics = process_file_predictions(
                model, file_path, running_stats_orig, running_stats_stretched
            )
            
            if gt is None:
                continue
                
            # Print metrics for this file
            print(f"EVA for model {epoch} is {metrics['eva_orig']:.4f}")
            print(f"RMSE for model {epoch} is {metrics['rmse_orig'] * 180:.2f}")
            print(f"RMSE for model {epoch} with stretched preds is {metrics['rmse_stretched'] * 180:.2f}")
            print(f"EVA for model {epoch} with stretched preds is {metrics['eva_stretched']:.4f}")
            
            # Plot stretched predictions
            time_axis = [x * (config["BINSIZE_MS"] / 1000) for x in range(len(preds_stretched))]
            ax.plot(time_axis, preds_stretched * 180, 
                   label=f"E:{epoch}, L:{metrics['rmse_orig']:.2f}")
        
        # Set up legend
        l = ax.legend(loc='best')
        if len(model_files) == 0:
            print("No models found! just plotting ground truth.")
        else:
            # Format legend text
            if len(l.get_texts()) > 2:
                l.get_texts()[-2].set_text(f"Best: {l.get_texts()[-2].get_text()}")
                l.get_texts()[-1].set_text(f"Last: {l.get_texts()[-1].get_text()}")
            else:
                l.get_texts()[-1].set_text(f"Best/Last: {l.get_texts()[-1].get_text()}")
    
# In plot_predictions function, update the section that prints cumulative metrics:
    # Print cumulative metrics
    if running_stats_orig.n_samples > 0:
        cum_eva_orig = running_stats_orig.get_eva()
        cum_eva_stretched = running_stats_stretched.get_eva()
        cum_rmse_orig = running_stats_orig.get_rmse()
        cum_rmse_stretched = running_stats_stretched.get_rmse()
        
        print("\n" + "=" * 50)
        print(f"CUMULATIVE EVA across all files: {cum_eva_orig:.4f}")
        print(f"CUMULATIVE RMSE across all files: {cum_rmse_orig * 180:.2f}")
        print(f"CUMULATIVE EVA with stretched preds across all files: {cum_eva_stretched:.4f}")
        print(f"CUMULATIVE RMSE with stretched preds across all files: {cum_rmse_stretched * 180:.2f}")
        print("=" * 50)
    
    plt.tight_layout()
    svg_path = os.path.join(config["IMAGES_DIR"], f"{dataset_name}.svg")
    fig.savefig(svg_path, format="svg")
    print(f"Saved plot to {svg_path}")
    plt.show()
    plt.close(fig)

In [ ]:
# filter files by dayset
day_filenames = []
night_filenames = []
day_files_list = "my_utils/csv_files/day-files.csv"
night_files_list = "my_utils/csv_files/night-files.csv"
with open(day_files_list, "r") as f, open(night_files_list, "r") as n:
    reader = csv.reader(f)
    for row in reader:
        day_filenames.extend(row)
    
    reader_n = csv.reader(n)
    for row in reader_n:
        night_filenames.extend(row)

day_train_files = [f for f in train_files if any(os.path.splitext(day)[0] in f for day in day_filenames)]
day_test_files = [f for f in test_files if any(os.path.splitext(day)[0] in f for day in day_filenames)]

night_train_files =  [f for f in train_files if any(os.path.splitext(day)[0] in f for day in night_filenames)]
night_test_files =  [f for f in test_files if any(os.path.splitext(day)[0] in f for day in night_filenames)]


In [ ]:
# See daytime stats
plot_predictions(model, day_test_files[:25], model_files, "Daytime_Test")

In [ ]:
# See nighttime stats
plot_predictions(model, night_test_files, model_files, "Night_Test")

In [ ]:
# plot every 20th one to save time
plot_predictions(model, test_files, model_files, "Test")

In [ ]:
# plot every 20th one to save time
plot_predictions(model, train_files[:15], model_files, "Train")

In [ ]:
# plot_predictions(model, whole_files[:int(len(whole_files) / 3):1], model_files, "Unedited1of3")
# plot_predictions(model, whole_files[int(len(whole_files) / 3):2 * int(len(whole_files) / 3):1], model_files, "Unedited2of3")
# plot_predictions(model, whole_files[2 * int(len(whole_files) / 3)::1], model_files, "Unedited3of3")